# 04 — MapReduce Aplicado a Voos

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 04/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Objetivos de aprendizagem
Calcular agregações com transparência sobre dados inválidos.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
MapReduce para totais por companhia, rotas frequentes e distâncias máximas.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
import pandas as pd
from collections import Counter
v = pd.read_csv('datasets/flights.csv')
assert {'dest','distance','origin'}.issubset(v.columns)
validos = v['dest'].dropna().astype(str).str.strip()
validos = validos[validos.ne('')]
contagens = Counter(validos)
referencia = v['dest'].dropna().astype(str).str.strip()
referencia = referencia[referencia.ne('')].value_counts().to_dict()
assert dict(contagens) == referencia
print('Destinos válidos:',sum(contagens.values()),'| ausentes/vazios:',len(v)-len(validos))
print('Top destinos:',sorted(contagens.items(),key=lambda p:(-p[1],p[0]))[:8])


Destinos válidos: 10000 | ausentes/vazios: 0
Top destinos: [('SFO', 787), ('LAX', 615), ('DEN', 586), ('PHX', 530), ('LAS', 520), ('ANC', 449), ('ORD', 439), ('SLC', 396)]


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
dist = v[['origin','dest','distance']].copy()
dist['distance'] = pd.to_numeric(dist['distance'], errors='coerce')
validos_dist = dist.dropna(subset=['origin','dest','distance'])
validos_dist = validos_dist[validos_dist['distance'] > 0]
assert not validos_dist.empty, 'Nenhuma distância positiva válida'
maximo = validos_dist.sort_values(['distance','origin','dest'],ascending=[False,True,True],kind='stable').iloc[0]
print('Distância máxima válida:',maximo.to_dict())
print('Registros excluídos da comparação:',len(dist)-len(validos_dist))
print('Distância não é tempo de voo; critério e desempate declarados.')


Distância máxima válida: {'origin': 'SEA', 'dest': 'MIA', 'distance': 2724}
Registros excluídos da comparação: 0
Distância não é tempo de voo; critério e desempate declarados.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 04 reuniu conceitos e verificações de **contagens e validação de dados**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** integridade de registros. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **integridade de registros** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **integridade de registros** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
